# Phase 2 — Policy RAG with Human-Readable Summary Chunks

Each policy is split at **semantic** boundaries, then rewritten as a short **analyst briefing** (title + bullets). Retrieval searches those briefings. Answers stay grounded in the original policy text.

**Pipeline:** PDFs → semantic chunks → AI summaries → embeddings → retrieve → grounded answer

## 1. Setup

Run this notebook from `notebook/Phase_2` with kernel **Python (zs-ai RAG)**.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, clear_output

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rag.chunking import run_chunking
from rag.pipeline import PolicyRAG
from rag.vectorstore import AzureRAGClient, index_corpora

azure = AzureRAGClient()
print("Chat model     :", azure.chat_model)
print("Embedding model:", azure.embedding_model)

## 2. Chunk, then rewrite each chunk as a briefing

Semantic split finds where the topic changes. The summarizer turns that excerpt into a scannable briefing and keeps the raw policy text as `source_text`.

In [ ]:
corpora = run_chunking(azure=azure)
semantic = corpora["semantic"]

comparison = pd.DataFrame([
    {
        "Strategy": name,
        "Chunks": len(rows),
        "Avg chars": round(sum(r["char_count"] for r in rows) / max(len(rows), 1), 1),
    }
    for name, rows in corpora.items()
])
display(comparison)

briefing_rows = []
for row in semantic:
    briefing_rows.append({
        "Plan": row["plan_type"],
        "Section": row["section"].replace("DOCUMENT HEADER", "Plan overview"),
        "Page": row["page"],
        "Policy briefing (what retrieval searches)": row["summary"],
        "Original excerpt": " ".join(row["source_text"].split())[:220] + "...",
    })
display(pd.DataFrame(briefing_rows))

## 3. Index the briefings

The vector store embeds the **briefing**, not the raw PDF dump. The original excerpt is stored as metadata for citations.

In [ ]:
counts = index_corpora(azure, strategies=("semantic",), records_by_strategy={"semantic": semantic})
display(pd.DataFrame([{"Collection": "policy_semantic", "Vectors": counts["semantic"]}]))

## 4. Ask a question

Retrieved rows show the briefing first. The model answers from the original policy excerpt.

In [ ]:
rag = PolicyRAG(azure=azure, strategy="semantic")

def evidence_df(chunks):
    return pd.DataFrame([
        {
            "Rank": c["rank"],
            "Similarity": round(c["similarity"], 3),
            "Citation": c["citation"],
            "Policy briefing": c.get("summary") or c["text"],
        }
        for c in chunks
    ])

def ask(question: str, plan: str = "auto", k: int = 4):
    use_plan_filter = plan not in ("", "all", "All plans")
    plan_type = None if plan in ("auto", "Auto", "", "all", "All plans") else plan
    result = rag.answer(
        question,
        top_k=k,
        plan_type=plan_type,
        use_plan_filter=use_plan_filter if plan != "auto" else True,
    )
    display(Markdown(f"**Question:** {result['question']}"))
    display(Markdown(f"**Plan filter:** {result['plan_filter'] or 'all plans'}"))
    display(Markdown(f"**Answer:** {result['answer']}"))
    display(evidence_df(result["retrieved"]))
    return result

ask("For a Gold PPO member, does the 11th physical therapy visit require prior authorization?")

## 5. Interactive ask box

Type a question and click **Ask**. You can also run `ask("your question")` in a new cell.

In [ ]:
import ipywidgets as widgets

question_box = widgets.Textarea(
    value="How many chiropractic visits are covered annually under Silver HMO?",
    layout=widgets.Layout(width="100%", height="80px"),
)
plan_box = widgets.Dropdown(
    options=[("Auto from question", "auto"), ("Gold PPO", "Gold PPO"), ("Silver HMO", "Silver HMO"), ("All plans", "all")],
    value="auto",
    description="Plan:",
)
ask_button = widgets.Button(description="Ask", button_style="primary")
out = widgets.Output()

def on_ask(_):
    q = question_box.value.strip()
    if not q:
        return
    with out:
        clear_output()
        ask(q, plan=plan_box.value)

ask_button.on_click(on_ask)
display(widgets.VBox([
    widgets.HTML("<h3>Healthcare Policy Assistant</h3>"),
    question_box,
    widgets.HBox([plan_box, ask_button]),
    out,
]))